In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [3]:
tools

[StructuredTool(name='search-flight', description='# Search for flights\n\nSearches Kiwi.com for available flights between two locations for the given dates and passengers. City or airport names are resolved automatically, so call this whenever the user wants to search for flights — whether they gave IATA codes or just place names.\n\n## Result shape\n\nReturns `{ query, currency, passengers, resultsCount, itineraries, searchTimeMs }`. Each item in `itineraries` has:\n- `tripType` — `"oneway"` or `"return"`\n- `price` (number) and `priceFormatted` (e.g. "123 EUR")\n- `totalDurationSeconds`\n- `bookingUrl` — the link to book the flight\n- `imageId` — destination city id for a hero photo (https://images.kiwi.com/photos/600x600/{imageId}.jpg)\n- `baggage` — total included baggage across all travelers: `{ personalItem, cabinBag, checkedBag }` (counts)\n- `outbound` (and `inbound` for return flights), each a leg with: `route` (list of airport codes including layovers, e.g. ["PRG","MAD","BCN

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [5]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31st")]},
    config
    )

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on March 31st', additional_kwargs={}, response_metadata={}, id='1a1546e3-1afd-402c-a3ca-7621163d2ea4'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 661, 'prompt_tokens': 2714, 'total_tokens': 3375, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 576, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Eje5uFMN5m7dHKuzHT28JGRtEAEdU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1d35f-21cd-79a0-8ad0-3ceab543d60c-0', tool_calls=[{'name': 'search-flight', 'args': {'flyFrom': 'San Francisco', 'flyTo': 'Tokyo', 'departureDate': '31/03/2027

In [7]:
print(response["messages"][-1].content)

Here are nonstop direct flights from San Francisco (SFO) to Tokyo (NRT or HND) on March 31, 2027. All options are direct economy.

| Itinerary | Route | Times (Outbound) | Cabin | Price | Booking |
| --- | --- | --- | --- | --- | --- |
| Best price | SFO → NRT | 31 Mar 12:20 → 01 Apr 15:25 (11h 05m) | Economy | 467 EUR | https://kiwi.com/u/84qnhq |
| Direct option | SFO → HND | 31 Mar 01:45 → 01 Apr 05:00 (11h 15m) | Economy | 536 EUR | https://kiwi.com/u/d6xnb3 |
| Direct option | SFO → NRT | 31 Mar 16:50 → 01 Apr 20:00 (11h 10m) | Economy | 567 EUR | https://kiwi.com/u/4d587b |
| Direct option | SFO → NRT | 31 Mar 12:20 → 01 Apr 15:25 (11h 05m) | Economy | 1002 EUR | https://kiwi.com/u/wvh724g |
| Direct option | SFO → NRT | 31 Mar 13:30 → 01 Apr 16:30 (11h 00m) | Economy | 3180 EUR | https://kiwi.com/u/ykk2qr3 |
| Direct option | SFO → HND | 31 Mar 11:55 → 01 Apr 15:05 (11h 10m) | Economy | 3180 EUR | https://kiwi.com/u/xck2np6 |
| Direct option | SFO → NRT | 31 Mar 13:30 → 01 Apr 1